# 🥉 Arquitetura Medalhão: Camada Bronze (Raw Data)

**Projeto:** Foodhunter (RAG System para Recomendação de Restaurantes)
**Objetivo desta camada:** A camada Bronze atua como o *Data Lake* inicial. Ela armazena os dados brutos (raw) exatamente como foram extraídos da fonte, sem nenhuma transformação, filtro ou limpeza. Ela é a nossa "fonte da verdade" histórica.

**Neste Notebook, vamos:**
1. Ingerir os dados brutos (`restaurants_with_embeddings.csv`).
2. Entender a volumetria e o schema dos dados.
3. Identificar sujeiras, valores nulos e anomalias que justifiquem as transformações da próxima etapa (Camada Silver).

In [3]:
# Importação das bibliotecas padrão para exploração
import pandas as pd
import numpy as np
import os
import json

# Configuração para o Pandas mostrar todas as colunas sem cortar
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Ingestão e Leitura dos Dados Brutos
Vamos carregar o arquivo CSV garantindo que o caminho dinâmico funcione independentemente de onde o notebook está sendo executado.

In [2]:
# Descobre o diretório atual do notebook (pasta bronze)
diretorio_atual = os.getcwd()
diretorio_raiz = os.path.dirname(diretorio_atual)

# Monta o caminho exato para o arquivo bruto
# Como o CSV está na pasta bronze na sua árvore, lemos direto dela
caminho_bronze = os.path.join(diretorio_atual, 'restaurants_with_embeddings.csv')

print(f"Lendo dados brutos de: {caminho_bronze}")
df_bronze = pd.read_csv(caminho_bronze)

# Exibe as 3 primeiras linhas para uma checagem visual rápida
display(df_bronze.head(3))

Lendo dados brutos de: c:\Faculdade - IA\Projeto AC2\RAG-Sapato\bronze\restaurants_with_embeddings.csv


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours,price,description,embedding
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'RestaurantsPriceRange2': '1', 'GoodForMeal': None, 'WiFi': ""u'...",Restaurants Food Bubble Tea Coffee & Tea Bakeries,"{'Wednesday': '7:0-20:0', 'Thursday': '7:0-20:0', 'Friday': '7:0-21:0', 'Saturday': '7:0-21:0', ...",1,Restaurants Food Bubble Tea Coffee & Tea Bakeries price:1,"[0.0035318147856742144, -0.008946051821112633, 0.04717320203781128, 0.03231402859091759, -0.0632..."
1,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"{'RestaurantsDelivery': 'True', 'RestaurantsPriceRange2': '1', 'GoodForMeal': None, 'WiFi': ""u'n...",Burgers Fast Food Sandwiches Food Ice Cream & Frozen Yogurt Restaurants,"{'Wednesday': '6:0-22:0', 'Thursday': '6:0-22:0', 'Friday': '9:0-0:0', 'Saturday': '9:0-22:0', '...",1,Burgers Fast Food Sandwiches Food Ice Cream & Frozen Yogurt Restaurants price:1,"[-0.005798297934234142, 0.013337074778974056, 0.05501795932650566, 0.03697891905903816, -0.00771..."
2,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,1,"{'RestaurantsDelivery': 'True', 'RestaurantsPriceRange2': '1', 'GoodForMeal': None, 'WiFi': ""u'n...",Ice Cream & Frozen Yogurt Fast Food Burgers Restaurants Food,"{'Wednesday': '6:0-21:0', 'Thursday': '6:0-16:0', 'Friday': '6:0-16:0', 'Saturday': '6:0-17:0', ...",1,Ice Cream & Frozen Yogurt Fast Food Burgers Restaurants Food price:1,"[0.01703026331961155, 0.0003804313892032951, 0.06728323549032211, 0.04034954681992531, -0.003436..."


## 2. Análise Estrutural e Tipagem (Schema)
Antes de limparmos qualquer coisa, precisamos entender o que o provedor de dados nos entregou. Quais são os tipos das colunas? Existem dados nulos?

In [4]:
# Informações gerais do DataFrame (tipos de dados e contagem de não-nulos)
print("=== SCHEMA DO DATASET BRUTO ===")
df_bronze.info()

print("\n=== CONTAGEM DE VALORES NULOS ===")
display(df_bronze.isnull().sum())

=== SCHEMA DO DATASET BRUTO ===
<class 'pandas.DataFrame'>
RangeIndex: 34987 entries, 0 to 34986
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   business_id   34987 non-null  str    
 1   name          34987 non-null  str    
 2   address       34662 non-null  str    
 3   city          34987 non-null  str    
 4   state         34987 non-null  str    
 5   postal_code   34983 non-null  str    
 6   latitude      34987 non-null  float64
 7   longitude     34987 non-null  float64
 8   stars         34987 non-null  float64
 9   review_count  34987 non-null  int64  
 10  is_open       34987 non-null  int64  
 11  attributes    34987 non-null  str    
 12  categories    34987 non-null  str    
 13  hours         31617 non-null  str    
 14  price         34979 non-null  str    
 15  description   34987 non-null  str    
 16  embedding     34987 non-null  str    
dtypes: float64(3), int64(2), str(12)
memory usage: 3

business_id        0
name               0
address          325
city               0
state              0
postal_code        4
latitude           0
longitude          0
stars              0
review_count       0
is_open            0
attributes         0
categories         0
hours           3370
price              8
description        0
embedding          0
dtype: int64

## 3. Identificação de Dívidas Técnicas (Por que precisamos da Silver?)
A camada Bronze é "suja" por natureza. Vamos inspecionar duas colunas críticas que o sistema RAG precisa, mas que vieram em formatos problemáticos: `attributes` e `embedding`.

### 3.1. O Problema da Coluna de Atributos (JSON Stringificado)
A coluna `attributes` contém dicionários, mas o CSV os salvou como strings mal formatadas (com aspas duplas vazadas e caracteres de escape).

In [5]:
# Pegando um exemplo aleatório da coluna de atributos
exemplo_atributo = df_bronze['attributes'].dropna().iloc[0]

print(f"TIPO DA VARIÁVEL: {type(exemplo_atributo)}")
print(f"CONTEÚDO BRUTO:\n{exemplo_atributo}")

# Tentar acessar como dicionário vai gerar erro, pois é uma string.
# A Camada Silver será responsável por fazer o 'parse' dessa string para um dict real usando ast.literal_eval.

TIPO DA VARIÁVEL: <class 'str'>
CONTEÚDO BRUTO:
{'RestaurantsDelivery': 'False', 'RestaurantsPriceRange2': '1', 'GoodForMeal': None, 'WiFi': "u'free'", 'WheelchairAccessible': None, 'GoodForKids': None, 'NoiseLevel': None, 'RestaurantsGoodForGroups': None, 'Alcohol': "u'none'", 'RestaurantsTableService': None, 'Caters': 'True', 'RestaurantsTakeOut': 'True', 'OutdoorSeating': 'False', 'Ambience': None, 'RestaurantsReservations': None, 'BikeParking': 'True', 'BusinessAcceptsCreditCards': 'False', 'RestaurantsAttire': None, 'HasTV': None, 'BusinessParking': "{'garage': False, 'street': True, 'validated': False, 'lot': False, 'valet': False}", 'BYOBCorkage': None, 'Corkage': None, 'DogsAllowed': None, 'ByAppointmentOnly': 'False', 'CoatCheck': None, 'HappyHour': None, 'Music': None, 'GoodForDancing': None, 'BestNights': None, 'HairSpecializesIn': None, 'BusinessAcceptsBitcoin': None, 'Smoking': None, 'AcceptsInsurance': None, 'DriveThru': None, 'BYOB': None, 'Open24Hours': None, 'Restauran

### 3.2. O Problema do Vetor de Embeddings
O coração do nosso RAG é a coluna `embedding`. O Banco de Dados Vetorial espera uma Lista de Floats, mas o CSV transformou essa lista inteira em uma única String gigante.

In [ ]:
# Pegando um exemplo do embedding
exemplo_embedding = df_bronze['embedding'].iloc[0]

print(f"TIPO DA VARIÁVEL: {type(exemplo_embedding)}")
print(f"TAMANHO DA STRING: {len(exemplo_embedding)} caracteres")
print(f"PRIMEIROS 50 CARACTERES: {exemplo_embedding[:50]}...")

# Conclusão: A Camada Silver precisará converter essas Strings em Listas nativas do Python
# e salvar o resultado em formato .parquet, para não perdermos o tipo da variável novamente.

## 4. Conclusão da Camada Bronze
A ingestão foi bem-sucedida. O dataset contém metadados ricos sobre restaurantes e embeddings pré-calculados.

**Deveres para o pipeline da Camada Silver (`silver.py`):**
* [x] Filtrar apenas restaurantes em operação (`is_open == 1`).
* [x] Converter a coluna `attributes` de String para Dicionário Python.
* [x] Extrair features estruturadas do dicionário (ex: `has_delivery`, `price_range`, `has_outdoor`).
* [x] Converter a coluna `embedding` de String para Listas nativas.
* [x] Descartar colunas desnecessárias.
* [x] Salvar o resultado final em formato columnar (`.parquet`) para preservar os *data types*.